In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
from models.linear_probes import linear_probe, linear_probe_tuned,linear_probe_tuned_smote
from models.feature_generation import build_feature_bank, extract_encoder, extract_feature,pool_features
from preprocessing.dataset import PipistrelleDataset
from evaluation.metrics import compute_cv_stats,plot_model_comparison,label_confusion
import pandas as pd
from sklearn.metrics import average_precision_score
from evaluation.metrics import compute_metrics,compile_results,generate_metrics_table2
from evaluation.metrics import plot_comprehensive_calibration
from models.MLP_balancing import balancing_mlp
import pickle
from pathlib import Path
import os
from models.mlsmote import get_minority_instace,MLSMOTE

c:\Users\artem\AppData\Local\miniconda3\envs\playground\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
c:\Users\artem\AppData\Local\miniconda3\envs\playground\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
c:\Users\artem\AppData\Local\m

In [3]:
dir = Path(os.getcwd()).resolve().parent 
path = str(dir / "models" / "features")
X_eff0 = np.load(path + "\\X_features2_not_normalized.npy")
X_NLM = np.load(path + "\\X_features_NLM.npy")
X_per2 = np.load(path + "\\perch_features.npz")['features']
y = np.load(path + "\\Y_labels2_not_normalized.npy")
label_names = ['Type A', 'Type B', 'Type C', 'Type D', 'Echo']

In [4]:
print(X_eff0.shape, X_NLM.shape, X_per2.shape, y.shape)

(284, 1280, 4, 32) (284, 496, 768) (284, 1536) (284, 5)


In [5]:
X_eff0_pooled = pool_features(X_eff0, windows=False, window_pooled=True, method='mean',encoder="effnetb0")
X_NLM_pooled = pool_features(X_NLM, windows=False, window_pooled=True, method='mean',encoder="NLM_BEATs")
X_per2_pooled = X_per2

In [15]:
X_sub, y_sub = get_minority_instace(pd.DataFrame(X_eff0_pooled), pd.DataFrame(y))   #Getting minority instance of that datframe
X_res,y_res =MLSMOTE(X_sub, y_sub, 100)
X_eff0_pooled2 = pd.concat([pd.DataFrame(X_eff0_pooled), X_res], axis=0)
y_new = pd.concat([pd.DataFrame(y), y_res], axis=0)
X_eff0_pooled2 = np.array(X_eff0_pooled2)
y_new = np.array(y_new)

In [16]:
eff0_all_results = linear_probe_tuned(X_eff0_pooled2, y_new, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

Starting Trial 1/5 with random_state=42...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: MLP
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Prevalence guesser
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
Starting Trial 2/5 with random_state=43...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3

In [18]:
label_names = ['Type A', 'Type B', 'Type C', 'Type D','Echo']

eff0_results = compile_results(eff0_all_results,label_names=label_names,encoder="eff")

results = eff0_results 
#print(pd.DataFrame(results))
print(generate_metrics_table2(results,label_names=label_names)[1])

                    Model      Type A AP      Type B AP      Type C AP  \
0       eff Random Forest  0.960 ± 0.010  0.892 ± 0.022  0.900 ± 0.010   
1                 eff SVM  0.881 ± 0.044  0.891 ± 0.020  0.879 ± 0.016   
2  eff Prevalence guesser  0.130 ± 0.009  0.219 ± 0.009  0.415 ± 0.027   
3                 eff MLP  0.940 ± 0.018  0.917 ± 0.028  0.895 ± 0.012   

       Type D AP        Echo AP  
0  0.993 ± 0.003  0.992 ± 0.003  
1  0.987 ± 0.012  0.985 ± 0.005  
2  0.370 ± 0.024  0.818 ± 0.009  
3  0.994 ± 0.002  0.987 ± 0.005  


In [22]:
eff0_all_results2 = linear_probe_tuned_smote(X_eff0_pooled, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

Starting Trial 1/5 with random_state=42...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: MLP
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Prevalence guesser
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
Starting Trial 2/5 with random_state=43...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3

In [25]:
eff0_results2= compile_results(eff0_all_results2,label_names=label_names,encoder="eff")

results2= eff0_results2
#print(pd.DataFrame(results))
print(generate_metrics_table2(results2,label_names=label_names)[1])

                    Model      Type A AP      Type B AP      Type C AP  \
0       eff Random Forest  0.449 ± 0.044  0.826 ± 0.014  0.884 ± 0.011   
1                 eff SVM  0.402 ± 0.052  0.865 ± 0.025  0.867 ± 0.010   
2  eff Prevalence guesser  0.081 ± 0.045  0.263 ± 0.033  0.590 ± 0.033   
3                 eff MLP  0.476 ± 0.055  0.855 ± 0.017  0.877 ± 0.011   

       Type D AP        Echo AP  
0  0.827 ± 0.022  0.978 ± 0.003  
1  0.831 ± 0.023  0.971 ± 0.008  
2  0.136 ± 0.022  0.826 ± 0.024  
3  0.855 ± 0.016  0.970 ± 0.008  


In [ ]:
X_concat = np.concatenate((X_eff0_pooled,X_per2_pooled,X_NLM_pooled), axis = 1)
print(X_concat.shape)

(284, 3584)


In [10]:
concat_all_results = linear_probe_tuned(X_concat, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

Starting Trial 1/5 with random_state=42...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: MLP
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Prevalence guesser
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
Starting Trial 2/5 with random_state=43...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3

In [13]:
label_names = ['Type A', 'Type B', 'Type C', 'Type D','Echo']

concat_results = compile_results(concat_all_results,label_names=label_names,encoder="eff")

print(generate_metrics_table2(concat_results,label_names=label_names)[1])

                    Model      Type A AP      Type B AP      Type C AP  \
0  eff Prevalence guesser  0.073 ± 0.010  0.239 ± 0.021  0.551 ± 0.028   
1       eff Random Forest  0.545 ± 0.050  0.841 ± 0.014  0.894 ± 0.009   
2                 eff MLP  0.617 ± 0.019  0.925 ± 0.012  0.903 ± 0.014   
3                 eff SVM  0.545 ± 0.038  0.898 ± 0.010  0.901 ± 0.018   

       Type D AP        Echo AP  
0  0.133 ± 0.015  0.826 ± 0.021  
1  0.855 ± 0.012  0.981 ± 0.001  
2  0.925 ± 0.011  0.973 ± 0.007  
3  0.946 ± 0.012  0.970 ± 0.006  


In [14]:
X_concat2 = np.concatenate((X_per2_pooled,X_NLM_pooled), axis = 1)

In [15]:
concat_all_results2 = linear_probe_tuned(X_concat2, y, n_split_out=5,n_split_in=5, num_trials=5,random_state=42)

Starting Trial 1/5 with random_state=42...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: MLP
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Prevalence guesser
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
Starting Trial 2/5 with random_state=43...
  Tuning and evaluating model: SVM
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3/5
    Evaluating fold 4/5
    Evaluating fold 5/5
  Tuning and evaluating model: Random Forest
    Evaluating fold 1/5
    Evaluating fold 2/5
    Evaluating fold 3

In [17]:
label_names = ['Type A', 'Type B', 'Type C', 'Type D','Echo']

concat_results2 = compile_results(concat_all_results2,label_names=label_names,encoder="eff")

print(generate_metrics_table2(concat_results2,label_names=label_names)[0])

                    Model      Macro-AUC Macro-AP (cmAP)  Brier Score ↓  \
0  eff Prevalence guesser  0.483 ± 0.019   0.364 ± 0.004  0.162 ± 0.001   
1       eff Random Forest  0.924 ± 0.004   0.832 ± 0.022  0.073 ± 0.002   
2                 eff MLP  0.940 ± 0.005   0.868 ± 0.007  0.064 ± 0.004   
3                 eff SVM  0.937 ± 0.006   0.856 ± 0.006  0.058 ± 0.001   

      Log-Loss ↓  
0  0.523 ± 0.007  
1  0.267 ± 0.014  
2  0.390 ± 0.023  
3  0.208 ± 0.005  
